In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import re

In [ ]:
# 1. Load and Preprocess the Dataset
def load_data(file_path):
    # Load the dataset (e.g., IMDB movie reviews dataset)
    df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')  # Using 'python' engine and skipping bad lines
    df.dropna(inplace=True)  # Drop any rows with missing values
    return df['review'], df['sentiment']  # Assuming 'review' and 'sentiment' columns

In [ ]:
# Clean the text
def clean_text(text):
    # Remove unwanted characters, numbers, and symbols
    text = re.sub(r"[^A-Za-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Tokenize and Pad Sequences
def preprocess_text(reviews, max_words=5000, max_len=200):
    reviews = [clean_text(review) for review in reviews]  # Clean the reviews
    tokenizer = Tokenizer(num_words=max_words)
    tokenizer.fit_on_texts(reviews)
    sequences = tokenizer.texts_to_sequences(reviews)
    padded_sequences = pad_sequences(sequences, maxlen=max_len)
    return padded_sequences, tokenizer



In [ ]:
# Encode Sentiments
def encode_labels(sentiments):
    sentiments = sentiments.map({'positive': 1, 'negative': 0}).values
    return sentiments

In [ ]:
# Load Data
file_path = 'IMDB Dataset.csv'  # <-- Provide the correct path to the dataset
reviews, sentiments = load_data(file_path)

In [ ]:
# Preprocess Text Data
max_words = 5000  # Consider the top 5000 words
max_len = 200  # Pad or truncate reviews to 200 words
X, tokenizer = preprocess_text(reviews, max_words=max_words, max_len=max_len)

In [ ]:
# Encode Sentiments (positive -> 1, negative -> 0)
y = encode_labels(sentiments)

# Split into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 2. Define the LSTM Model
model = Sequential()

# Modify the embedding dimensions and experiment with LSTM configurations ---
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))  # <-- Modify 'output_dim'
model.add(Bidirectional(LSTM(units=64, return_sequences=False)))  # <-- Experiment with 'units' and add Dropout if necessary

model.add(Dense(1, activation='sigmoid'))  # Output layer for binary classification

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Train the Model
#  Modify 'epochs' and 'batch_size' to see how they impact training time and model accuracy ---
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=1)  # <-- Experiment with 'epochs' and 'batch_size'


Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 261s 206ms/step - accuracy: 0.7434 - loss: 0.5066 - val_accuracy: 0.8700 - val_loss: 0.3083
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 264s 208ms/step - accuracy: 0.8835 - loss: 0.2926 - val_accuracy: 0.8814 - val_loss: 0.2864
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 264s 210ms/step - accuracy: 0.9103 - loss: 0.2364 - val_accuracy: 0.8901 - val_loss: 0.2684
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 320s 208ms/step - accuracy: 0.9269 - loss: 0.1912 - val_accuracy: 0.8881 - val_loss: 0.2816
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 256s 205ms/step - accuracy: 0.9441 - loss: 0.1532 - val_accuracy: 0.8854 - val_loss: 0.2960
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 267s 209ms/step - accuracy: 0.9569 - loss: 0.1177 - val_accuracy: 0.8830 - val_loss: 0.3460
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 262s 209ms/step - accuracy: 0.9664 - loss: 0.0932 - val_accuracy: 0.8814 - val_loss: 0.3763
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 263s 209ms/step - accuracy: 0.9

In [ ]:
# 4. Evaluate the Model
y_pred = (model.predict(X_test) > 0.5).astype("int32")


313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step


In [ ]:
# Calculate Accuracy and F1-Score
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print(f'F1-Score: {f1:.4f}')


#  Analyze the accuracy and F1-score. Consider modifying the model architecture or hyperparameters to improve performance ---

Accuracy: 0.8802
F1-Score: 0.8790


Unidirectional LSTM

Processes text only forward (from the first word to the last).

Captures past context but not future context.

Faster and less computationally expensive.

May miss subtle dependencies if important information appears later in the sequence.

Bidirectional LSTM

Processes text in both forward and backward directions.

Captures both past and future context.

Typically improves performance in NLP tasks where long-range dependencies matter (like sentiment analysis).

More computationally expensive (roughly 2x parameters)

Unidirectional LSTM	~0.86–0.87	~0.86	Slightly lower performance because it only considers past context.
Bidirectional LSTM	0.8802	0.8790	Better at capturing full sentence meaning since it considers both past & future words.